# Lesson 03 - एजंटिक डिझाइन पॅटर्न्स

या धड्यात, आम्ही प्रभावी AI एजंट बनवण्यासाठी तीन मूलभूत डिझाइन पॅटर्न्सचा अभ्यास करू:

1. **स्पष्ट एजंट सूचना** — एजंट वर्तन मार्गदर्शन करण्यासाठी अचूक, भूमिका-व्याख्यात्मक प्रॉम्प्ट तयार करणे
2. **Pydantic मॉडेल्ससह संरचित आउटपुट** — एजंट्सला पूर्वानुमानित, मान्यताप्राप्त डेटा परत करण्यास सुनिश्चित करणे
3. **एकल जबाबदारी एजंट्स** — लक्ष केंद्रीत एजंट्स डिझाइन करणे जे प्रत्येक एक काम चांगल्या प्रकारे करतात

आम्ही प्रत्येक पॅटर्न एका **प्रवास स्थळ शिफारसकर्त्या** परिस्थितीवर लागू करू, प्रगतिशीलपणे अशी प्रणाली तयार करत जिथे स्थळे सुचवता येतील, उपलब्धता तपासता येईल, आणि लॉजिस्टिक्स हाताळता येतील.


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Pattern 1: स्पष्ट एजंट सूचना

सर्वात प्रभावी नमुना म्हणजे सर्वात सोपा: तुमच्या एजंटसाठी स्पष्ट, तपशीलवार सूचना लिहिणे.

चांगल्या सूचनांमध्ये या गोष्टींचा समावेश असतो:
- **कोण** एजंट आहे (व्यक्तिमत्व आणि टोन)
- **काय** करावे (टप्प्याटप्प्याने जबाबदाऱ्या)
- **कसे** वागावे (मर्यादा आणि शैली)

खाली, आम्ही एक प्रवास कन्सियर्ज एजंट तयार करतो ज्याच्या प्रत्येक प्रतिसादात स्पष्ट सूचना असतात ज्या त्याच्या प्रत्येक उत्तराला घडवतात.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Pydantic मॉडेल्ससह संरचित आउटपुट

मोकळा मजकूर संभाषणासाठी उपयुक्त असतो, परंतु खालील सिस्टमना संरचित डेटा आवश्यक असतो.
**Pydantic मॉडेल्स** आणि **टूल फंक्शन** यांची जोडी लावून आपण:

- एजंटच्या आउटपुटसाठी अचूक स्कीमा परिभाषित करू शकतो
- प्रतिसाद आपोआप तपासू शकतो
- एजंटचे निकाल अॅप्लिकेशन लॉजिकमध्ये विश्वासार्हपणे समाकलित करू शकतो

आम्ही एक टूल देखील सादर करतो जे गंतव्य तपशील परत करते जेणेकरून एजंटच्या शिफारसी वास्तविक डेटावर आधारभूत असतील.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## नमुना 3: एकल जबाबदारी एजंट्स

संकुल कामांना एकाधिक केंद्रित एजंट्समध्ये विभाजित केल्याने फायदा होतो, ज्यापैकी प्रत्येकाची एकच जबाबदारी असते:

- एक **गंतव्य तज्ञ** जो ठिकाणे आणि उपलब्धता याबाबत माहिती ठेवतो
- एक **लॉजिस्टिक्स नियोजक** जो फ्लाइट्स, हॉटेल्स, आणि सहलींचे आयोजन करतो

हे सॉफ्टवेअर अभियांत्रिकीच्या *व्यवहार वेगळेपण* तत्वाशी जुळते — प्रत्येक एजंट स्वतंत्रपणे तपासणे, देखभाल करणे आणि सुधारणा करणे सोपे असते.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## सारांश

या धड्यात आपण ट्रॅव्हल शिफारस करणाऱ्या परिस्थितीसाठी तीन एजंटिक डिझाइन पॅटर्न लागू केले:

| पॅटर्न | मुख्य कल्पना | फायदा |
|---|---|---|
| **स्पष्ट सूचना** | व्यक्तिमत्व, जबाबदाऱ्या, आणि मर्यादा सुरुवातीला निश्चित करा | सातत्यपूर्ण, ब्रँडशी सुसंगत एजंट वर्तन |
| **रचनेतली आऊटपुट** | प्रतिसाद स्वरुपासाठी Pydantic मॉडेल्स वापरा | प्रमाणीकरण केलेले, मशीन वाचण्याजोगे निकाल |
| **एकल जबाबदारी** | प्रत्येक एजंटला एक लक्ष केंद्रीत काम द्या | चाचणी, देखभाल आणि संयोजन सुलभ होते |

हे पॅटर्न नैसर्गिकरित्या एकत्र येतात — तुम्ही एकल जबाबदारी असलेल्या एजंटमध्ये स्पष्ट सूचना आणि रचनेतली आऊटपुट एकत्र करून मजबूत, उत्पादनासाठी तयार प्रणाली तयार करू शकता.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
